# 08 - Modelos base
En este notebook probamos los dos modelos base (`dummy` y `logistic_regression`) y las tres métricas (F1 macro, exactitud balanceada y kappa ponderado cuadrático) con los datos reales (`data/processed/streets_unified.csv`).

Para cada partición (dentro de cada ciudad, entre ciudades y mixta) entrenamos con el conjunto de train y evaluamos con el de test. Después resumimos con la media y la desviación típica entre folds.

## Carga de datos

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from urban_accessibility_ml.config import METRICS, PROCESSED_DATA_DIR
from urban_accessibility_ml.modeling.evaluate import evaluate, summarize_folds
from urban_accessibility_ml.modeling.models import build_model
from urban_accessibility_ml.plotting.palettes import SERIE_BASE
from urban_accessibility_ml.plotting.plots import SERIES, save_figure, use_style
from urban_accessibility_ml.preprocessing.features import get_xy
from urban_accessibility_ml.preprocessing.split import (
    ID_COL, split_cross_city, split_mixed, split_within_city,
)

MODELS = ["dummy", "logistic_regression"]
FEATURE_SET = "completo"
PROFILE = "global"
CITIES = ["Albacete", "Algeciras"]
MODEL_COLORS = {"dummy": SERIE_BASE, "logistic_regression": SERIES[3]}
METRIC_TITLES = {
    "f1_macro": "F1 macro",
    "balanced_accuracy": "Exactitud balanceada",
    "quadratic_weighted_kappa": "Kappa ponderado cuadrático",
}

use_style()

# Indexed by key_tramo: the splits store those ids
df = pd.read_csv(PROCESSED_DATA_DIR / "streets_unified.csv").set_index(ID_COL, drop=False)
df.shape

## Particiones
Obtenemos los `Split`: dentro de cada ciudad (5 pliegues), entre ciudades (Albacete-Algeciras y Algeciras-Albacete) y mixta (5 pliegues).

In [ ]:
splits = (
    [s for city in CITIES for s in split_within_city(df, city)]
    + split_cross_city(df, "Albacete", "Algeciras")
    + split_cross_city(df, "Algeciras", "Albacete")
    + split_mixed(df)
)
pd.Series([(s.strategy, s.name) for s in splits]).value_counts().sort_index()

## Entradas y clase
Se obienen una sola vez, puesto que se van a usar las mismas para todas las particiones y modelos.

In [ ]:
X, y = get_xy(df, FEATURE_SET, PROFILE)
print(list(X.columns))
y.value_counts().sort_index()

## Entrenamiento y evaluación
Vamos a realizar el entrenamiento usando los modelos dummy y regresión logística.

Una vez realizado, vamos a calcular las métricas f1_macro, balanced_accuracy y quadratic_weighted_kappa.

Cada fila es un modelo evaluado en un pliegue, en las estrategias: `within_city`, `cross_city` y `mixed`.

In [ ]:
rows = []
for split in splits:
    X_train, y_train = X.loc[split.train_ids], y.loc[split.train_ids]
    X_test, y_test = X.loc[split.test_ids], y.loc[split.test_ids]
    for model_name in MODELS:
        model = build_model(model_name).fit(X_train, y_train)
        rows.append({
            "strategy": split.strategy, "name": split.name, "fold": split.fold,
            "model": model_name,
            **evaluate(y_test, model.predict(X_test)),
        })
results = pd.DataFrame(rows)
results.round(3)

## Medias y desviaciones típicas de los pliegues
Calculamos después la media y desviación típica entre pliegues para cada partición y modelo, exceptuando en `cross_city`, que no hay pliegues.

Debería haber dos condiciones que se deberían cumplir:
- El modelo `dummy` tendría que tener un 0,33 de media o similar en exactitud balanceada.
- La regresión logística debería superar al modelo `dummy` en todas las particiones.

In [ ]:
summary = summarize_folds(results, ["strategy", "name", "model"])

mean = summary.pivot(index=["strategy", "name"], columns="model", values="balanced_accuracy_mean")
print("dummy mean:", mean["dummy"].round(2).tolist())
print("logistic_regression > dummy en todas:", bool((mean["logistic_regression"] > mean["dummy"]).all()))
summary.round(3)

## Visualización de métricas
Vamos a mostrar un gráfico por cada una de las 3 métricas utilizadas: la barra es la media entre pliegues y la línea la desviación típica. Cuanto mayor sea la barra y más cercana a 1 esté, mejor.

El `dummy` tiene un kappa ponderada cuadrática de 0, no es un error. Esto es debido que es el modelo más "tonto", e intenta siempre acertar según la configuración, la clase más frecuente.

In [ ]:
groups = summary[["strategy", "name"]].drop_duplicates()
labels = (groups["strategy"] + "\n" + groups["name"]).tolist()
ypos = np.arange(len(groups))
height = 0.38

fig, axes = plt.subplots(1, len(METRICS), figsize=(13, 4.2), sharey=True)
for ax, metric in zip(axes, METRICS):
    for i, model_name in enumerate(MODELS):
        s = groups.merge(summary[summary["model"] == model_name], on=["strategy", "name"])
        ax.barh(
            ypos + (i - 0.5) * height, s[f"{metric}_mean"], height=height,
            xerr=s[f"{metric}_std"].fillna(0), color=MODEL_COLORS[model_name], label=model_name,
            error_kw={"linewidth": 1},
        )
    ax.set_title(METRIC_TITLES[metric])
    ax.set_xlim(0, 1)
axes[0].set_yticks(ypos, labels)
axes[0].invert_yaxis()
handles, names = axes[0].get_legend_handles_labels()
fig.legend(handles, names, loc="lower center", ncol=len(MODELS), frameon=False)
fig.tight_layout(rect=(0, 0.06, 1, 1))
save_figure("metricas_modelos_base", "0.08-modelos-base", fig=fig)

También podemos ver que la estrategia de `cross_city` obtiene los peores resultados, por lo que parece que la transferencia entre ciudades es más complicada que en la propia ciudad.
Mezclando datos de ambas ciudades (estrategia `mixed`), parece mejorar bastante esos resultados.